### Log Reader

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3-coder:480b-cloud",
    temperature=0.5,
    max_tokens=250
)

c:\Users\Lenovo\Downloads\agentsforQA_code\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

#### Read the log file

In [ ]:
import os
from langchain.tools import tool
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

@tool
def summarize_logs() -> str:
    "Read and returns a summary of the first few lines from each .log file. DONT get into the details of them."
    log_dir = "./logs"
    db_path = "./qa_db"
    all_logs = []
    for file in os.listdir(log_dir):
        with open(os.path.join(log_dir, file)) as f:
            all_logs.extend([f"{file}: {line.strip()}" for line in f.readlines()[:50]])
    summary = "\n".join(all_logs) or "No logs found"

    #ToDo
    if not(os.path.exists(db_path) and os.path.isdir(db_path)):
        # spliting
        splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        chunks = splitter.create_documents([summary])
        
        # embedding
        embedding = OllamaEmbeddings(model="nomic-embed-text")
        db = Chroma.from_documents(chunks, embedding, persist_directory=db_path)
        db.persist()
        summary += "\n\n[Embedding done and DB Created]"
    else:
        summary += "\n\n[Embedding skipped:DB already Created]"
    
    return summary




In [ ]:
from langchain_classic.agents import initialize_agent, AgentType


@tool
def add_numbers(a: int, b: int) -> int:
    "Adding two given numbers and returns a result"
    return int(a) + int(b)

@tool
def substract_numbers(a: int, b: int) -> int:
    "Subtract two given numbers and returns a result"
    return int(a) - int(b)

@tool
def multiply_numbers(a: int, b: int) -> int:
    "Multiply two given numbers and returns a result"
    return int(a) * int(b)

tools = [add_numbers, substract_numbers, multiply_numbers, summarize_logs]

# Creating an AI Agent
agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
tool_by_name = {tool.name: tool for tool in tools}
tool_by_name

In [ ]:
query = "What are the warnings in the application"

result = agent.run(query)

print(result)